In [ ]:
import shutil
import sys

import dotenv
import polars as pl
from Bio import Entrez

sys.path.insert(0, "..")
from project import get_all, log

In [ ]:
dotenv.load_dotenv()

email = dotenv.get_key(dotenv.find_dotenv(), "ENTREZ_EMAIL")
if email is None:
    raise ValueError("ENTREZ_EMAIL not found in .env file or environment variables.")
Entrez.email = email
print(f"Entrez email set to: {email}")

api_key = dotenv.get_key(dotenv.find_dotenv(), "ENTREZ_API_KEY")
if api_key is None:
    print("ENTREZ_API_KEY not found in .env file, proceeding without it.")
else:
    print("Entrez API key set.")
    Entrez.api_key = api_key

In [ ]:
ALL = get_all()  # or get_all(["plants_plt"])
# ALL = {"plants_plt": {"solanaceae": get_info("plants_plt", "solanaceae")}}

REFETCH_LINEAGE = False  # True queries NCBI again even if <group>_lineage.tsv exists
QUERY_SIZE = 250  # taxids per Entrez request
LABEL_PREFIX = r"(?:PLT|MIT)-"  # stripped from UP/DOWN when building gene_pair names

for g, subs in ALL.items():
    print(f"{g}: {list(subs)}")

## Group files: copy from the atlas, fetch lineages

In [ ]:
for g, subs in ALL.items():
    p = next(iter(subs.values()))[
        "paths"
    ]  # group-level paths are the same for every subgroup
    src = p["atlas_source"]
    p["group_dir"].mkdir(parents=True, exist_ok=True)

    for name, dest in [
        (f"{g}.tsv", p["group_tsv"]),
        (f"{g}_summary_igr.tsv", p["group_atlas"]),
        (f"{g}.gdict", p["group_gdict"]),
        ("tree.nwk", p["group_tree"]),
    ]:
        shutil.copy(src / name, dest)
    log(f"{g}: copied group files from {src}")

    if p["group_lineage"].exists() and not REFETCH_LINEAGE:
        log(
            f"{g}: {p['group_lineage'].name} exists, skipping NCBI (REFETCH_LINEAGE = True to redo)"
        )
        continue

    tsv = pl.read_csv(p["group_tsv"], separator="\t")
    taxids = tsv["ncbi_taxid"].unique().to_list()
    taxid_lineage = {}
    for start in range(0, len(taxids), QUERY_SIZE):
        window = taxids[start : start + QUERY_SIZE]
        with Entrez.efetch(db="taxonomy", id=",".join(map(str, window))) as handle:
            for record in Entrez.read(handle):
                taxid_lineage[str(record["TaxId"])] = str(record["Lineage"])
        log(f"  {g}: lineages {start + len(window)}/{len(taxids)}")

    tsv = tsv.with_columns(
        pl.col("ncbi_taxid")
        .cast(pl.String)
        .replace_strict(taxid_lineage, default="Unknown")
        .alias("lineage")
    )
    n_unknown = tsv.filter(pl.col("lineage") == "Unknown").height
    tsv.write_csv(p["group_lineage"], separator="\t")
    log(
        f"{g}: {len(taxids)} taxids -> {p['group_lineage'].name} ({n_unknown} rows without a lineage)"
    )

## Subgroups: filter by lineage, drop QC-excluded ANs, one genome per taxid

In [ ]:
for g, subs in ALL.items():
    first = next(iter(subs.values()))["paths"]
    tsv = pl.read_csv(first["group_lineage"], separator="\t").with_columns(
        pl.col("lineage")
        .str.split(";")
        .list.eval(pl.element().str.strip_chars())
        .list.eval(pl.element().filter(pl.element() != ""))
        .alias("lineage_taxa")
    )
    atlas = pl.read_csv(first["group_atlas"], separator="\t").with_columns(
        pl.concat_str(
            pl.col("UP").str.replace_all(LABEL_PREFIX, ""),
            pl.col("DOWN").str.replace_all(LABEL_PREFIX, ""),
            pl.col("Polarity"),
            separator="_",
        ).alias("gene_pair")
    )

    for subg, info in subs.items():
        p = info["paths"]
        keywords = info["config"].get("lineage_keywords")
        if not keywords:
            log(f"{g}/{subg}: no lineage_keywords in {p['yaml'].name}, skipping")
            continue

        excluded = (
            pl.read_csv(p["qc_excluded"], separator="\t")["AN"].to_list()
            if p["qc_excluded"].exists()
            else []
        )

        subg_tsv = tsv.filter(
            pl.any_horizontal(pl.col("lineage_taxa").list.contains(k) for k in keywords)
        ).drop("lineage", "lineage_taxa")
        n_lineage = subg_tsv.height

        n_excl = subg_tsv.filter(pl.col("AN").is_in(excluded)).height
        subg_tsv = subg_tsv.filter(~pl.col("AN").is_in(excluded))

        # one genome per taxid: keep the longest
        before = subg_tsv.height
        subg_tsv = (
            subg_tsv.sort("Genome_length", descending=True)
            .unique(subset=["ncbi_taxid"], keep="first")
            .sort("AN")
        )

        p["dir"].mkdir(parents=True, exist_ok=True)
        subg_tsv.write_csv(p["tsv"], separator="\t")
        atlas.filter(pl.col("AN").is_in(subg_tsv["AN"].to_list())).write_csv(
            p["atlas"], separator="\t"
        )
        log(
            f"{g}/{subg}: {n_lineage} ANs match {keywords} | -{n_excl} QC-excluded"
            f" | -{before - subg_tsv.height} duplicate taxids | {subg_tsv.height} kept"
        )

## Subgroups: copy per-genome files into `source/`

In [ ]:
PER_GENOME = [
    ("genomes", "{an}.fasta"),
    ("gff3", "{an}.gff3"),
    ("igr_by_genome", "{an}_intergenic.fasta"),
]

for g, subs in ALL.items():
    for subg, info in subs.items():
        p = info["paths"]
        if not p["tsv"].exists():
            log(f"{g}/{subg}: no {p['tsv'].name}, skipping")
            continue
        ans = pl.read_csv(p["tsv"], separator="\t")["AN"].to_list()

        for key, pattern in PER_GENOME:
            dest_dir = p[key]
            dest_dir.mkdir(parents=True, exist_ok=True)
            missing = []
            for an in ans:
                src_file = p["atlas_source"] / an / pattern.format(an=an)
                if src_file.exists():
                    shutil.copy(src_file, dest_dir / src_file.name)
                else:
                    missing.append(an)
            # files left from ANs no longer in the subgroup
            stale = {f.name for f in dest_dir.iterdir()} - {
                pattern.format(an=an) for an in ans
            }
            log(
                f"{g}/{subg} {key}: {len(ans) - len(missing)}/{len(ans)} copied"
                + (
                    f" | missing: {missing[:5]}{'...' if len(missing) > 5 else ''}"
                    if missing
                    else ""
                )
                + (f" | {len(stale)} stale files from removed ANs" if stale else "")
            )